<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


**Dataset lineage:** EDA generates an 80-column feature matrix. The supplied course modeling snapshot contains 83 columns because it also dummy-encodes the three binary fields (GridFins, Reused and Legs). The ML notebook deliberately loads that supplied 83-column reference matrix to reproduce the report. Generated exports remain in data/processed and never replace data/reference.

## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


## Repository execution mode
This notebook parses a bundled copy of the historical Wikipedia revision dated 9 June 2021. The HTML was retrieved during repository preparation, so the notebook can run without Internet access.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
assert (PROJECT_ROOT / 'data').is_dir(), 'Open this notebook from the repository root or notebooks folder.'

# Install the root requirements.txt before running the notebook.

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def clean_cell(cell):
    # Remove citation markers without modifying the original table.
    clone = BeautifulSoup(str(cell), 'html.parser')
    for tag in clone.select('sup.reference'):
        tag.decompose()
    return unicodedata.normalize('NFKC', clone.get_text(' ', strip=True))

def date_time(table_cells):
    text = clean_cell(table_cells)
    date_match = re.search(r'\d{1,2}\s+[A-Za-z]+\s+\d{4}', text)
    time_match = re.search(r'\d{2}:\d{2}(?::\d{2})?', text)
    return [date_match.group(0) if date_match else text,
            time_match.group(0) if time_match else None]

def booster_version(table_cells):
    return clean_cell(table_cells)

def landing_status(table_cells):
    return clean_cell(table_cells)

def get_mass(table_cells):
    text = clean_cell(table_cells)
    match = re.search(r'[\d, .]+\s*kg', text)
    return match.group(0).strip() if match else text or None

def extract_column_from_header(row):
    name = clean_cell(row)
    return name if name and not name.isdigit() else None

To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
html_text = (PROJECT_ROOT / 'data/raw/wikipedia_launches_2021-06-09.html').read_text(encoding='utf-8')
print('Loaded the actual Wikipedia revision downloaded for this repository.')

Loaded the actual Wikipedia revision downloaded for this repository.


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
soup = BeautifulSoup(html_text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
print(soup.title.get_text(strip=True) if soup.title else 'Titre absent')

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
html_tables = soup.find_all('table')
print('Nombre de tableaux :', len(html_tables))

Nombre de tableaux : 29


Starting from the third table is our target table contains the actual launch records.


In [9]:
# Identify launch tables by their headers rather than a fragile table index.
launch_tables = []
for table in html_tables:
    first_row = table.find('tr')
    if first_row:
        names = [clean_cell(th).lower() for th in first_row.find_all('th')]
        if any('flight' in name for name in names) and any('payload' in name for name in names):
            launch_tables.append(table)
if not launch_tables:
    raise ValueError('Aucun tableau de lancements trouvé : vérifier la page reçue.')
first_launch_table = launch_tables[0]
print(first_launch_table.find('tr'))

<tr id="mwmw"><th id="mwnA" scope="col">Flight No.</th>
<th id="mwnQ" scope="col">Date and<br id="mwng"/>time (<a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">UTC</a>)</th>
<th id="mwoA" scope="col"><a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">Version,<br id="mwog"/>Booster</a> <sup about="#mwt70" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-booster-11"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"},"1":{"wt":"Falcon 9 first-stage boosters are designated with a construction serial number and an optional flight number when reused, e.g. B1021.1 and B1021.2 represent the two flights of booster [[B1021]]. Launches using reused boosters are denoted with a re

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []
for th in first_launch_table.find('tr').find_all('th'):
    name = extract_column_from_header(th)
    if name:
        column_names.append(name)

Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
launch_dict = {key: [] for key in ['Flight No.', 'Date', 'Time', 'Version Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
extracted_row = 0
for table in launch_tables:
    for tr in table.find_all('tr'):
        heading = tr.find('th', recursive=False)
        if heading is None:
            continue
        flight_number = clean_cell(heading)
        # Falcon Heavy entries use labels such as FH 1; retain Falcon 9 only.
        if not flight_number.isdigit():
            continue
        row = tr.find_all('td', recursive=False)
        if len(row) != 9:
            raise ValueError(f'Vol {flight_number}: {len(row)} cellules au lieu de 9.')
        date, time = date_time(row[0])
        values = [flight_number, date, time, booster_version(row[1]),
                  clean_cell(row[2]), clean_cell(row[3]), get_mass(row[4]),
                  clean_cell(row[5]), clean_cell(row[6]), clean_cell(row[7]),
                  landing_status(row[8])]
        for key, value in zip(launch_dict, values):
            launch_dict[key].append(value)
        extracted_row += 1
print('Lancements extraits :', extracted_row)

Lancements extraits : 121


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
df = pd.DataFrame(launch_dict)
assert not df.empty, 'Aucun lancement extrait.'
assert df['Flight No.'].is_unique, 'Numéros de vol dupliqués.'
display(df.head())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Flight No.       121 non-null    object
 1   Date             121 non-null    object
 2   Time             120 non-null    object
 3   Version Booster  121 non-null    object
 4   Launch site      121 non-null    object
 5   Payload          121 non-null    object
 6   Payload mass     119 non-null    object
 7   Orbit            121 non-null    object
 8   Customer         121 non-null    object
 9   Launch outcome   121 non-null    object
 10  Booster landing  121 non-null    object
dtypes: object(11)
memory usage: 10.5+ KB


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,None,LEO,SpaceX,Success,Failure (parachute)
1,2,8 December 2010,15:43,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),None,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,22 May 2012,07:44,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),525 kg,LEO ( ISS ),NASA ( COTS ),Success,No attempt
3,4,8 October 2012,00:35,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,1 March 2013,15:10,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.


In [15]:
df.to_csv(PROJECT_ROOT / 'data/processed/spacex_web_scraped.csv', index=False)
print(f'Export : spacex_web_scraped.csv ({len(df)} lignes)')

Export : spacex_web_scraped.csv (121 lignes)


### Utilisation
Exécuter les cellules dans l’ordre. La page utilisée est la révision historique du 9 juin 2021, afin de conserver le périmètre du TP. Une connexion Internet et l’accès à Wikipédia sont nécessaires. Les résultats ne sont pas préremplis.